# Security room access analysis
O arquivo `data/sample_access_events.csv` contém dados fictícios para demonstrar a análise. Para analisar dados reais, altere `DATA_FILE` para `access_events.csv` exportado pelo sistema e ajuste a data e matrícula.

In [1]:
import pandas as pd
# Parâmetros: ajuste a data e a matrícula para a consulta desejada.
ANALYSIS_DATE = '2026-10-26'
TARGET_EMPLOYEE = 'EMP001'
DATA_FILE = 'data/sample_access_events.csv'
events = pd.read_csv(DATA_FILE, parse_dates=['timestamp'])
events['day'] = events['timestamp'].dt.strftime('%Y-%m-%d')
daily = events[events['day'] == ANALYSIS_DATE].sort_values('timestamp')
daily.head()

                  timestamp employee_id  ...  outcome         day
0 2026-10-26 09:00:00+00:00      EMP001  ...  granted  2026-10-26
1 2026-10-26 10:15:00+00:00      EMP002  ...   denied  2026-10-26
2 2026-10-26 11:00:00+00:00         NaN  ...   denied  2026-10-26
3 2026-10-26 12:00:00+00:00      EMP001  ...  granted  2026-10-26
4 2026-10-26 14:30:00+00:00      EMP002  ...   denied  2026-10-26

[5 rows x 7 columns]

In [2]:
# Conta eventos de cada categoria no dia selecionado.
print('Entries:', int((daily['type'] == 'entry').sum()))
print('Exits:', int((daily['type'] == 'exit').sum()))
print('Denied attempts:', int((daily['type'] == 'access_denied').sum()))
print('Unknown-tag attempts:', int((daily['type'] == 'unknown_tag').sum()))

Entries: 1
Exits: 1
Denied attempts: 2
Unknown-tag attempts: 1


In [3]:
# Seleciona o colaborador e percorre eventos em ordem cronológica para parear entrada/saída.
person_events = daily[daily['employee_id'] == TARGET_EMPLOYEE]
# Entrada sem saída correspondente não entra no total; não há horário de término confiável.
open_entry = None
seconds_inside = 0
for row in person_events.itertuples():
    if row.type == 'entry':
        open_entry = row.timestamp
    elif row.type == 'exit' and open_entry is not None:
        seconds_inside += max(0, (row.timestamp - open_entry).total_seconds())
        open_entry = None
print(f'{TARGET_EMPLOYEE} time inside on {ANALYSIS_DATE}: {seconds_inside / 60:.1f} minutes')
if open_entry is not None:
    print('Note: unmatched entry; its duration is not included.')

EMP001 time inside on 2026-10-26: 180.0 minutes


In [4]:
# Agrupa tentativas negadas por matrícula e nome para encontrar as mais frequentes.
(daily[daily['type'] == 'access_denied']
 .groupby(['employee_id', 'employee_name'], dropna=False)
 .size().sort_values(ascending=False).rename('attempts').head(10))

employee_id  employee_name
EMP002       Bea              2
Name: attempts, dtype: int64